# YouTube Video Search

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook searches YouTube and exports structured video metadata for media research. Enter search terms, set a result limit, and download video titles, channels, view counts, durations, publication dates, and URLs as CSV or Excel.

The notebook uses the `scrapetube` library to query YouTube directly — no API key required. Results include all the metadata visible on a YouTube search results page, structured for qualitative analysis.

## Key Features

1. **No API Key Required**: Queries YouTube directly using the open-source `scrapetube` library
2. **Keyword Search**: Search across all of YouTube's indexed video content
3. **Configurable Result Count**: Fetch 10 to 200 results per search
4. **Rich Metadata**: Titles, channels, view counts, durations, publication dates, and direct URLs
5. **Structured Export**: Download results as CSV or styled Excel for further analysis
6. **Pairs with Transcript Fetcher**: Use this notebook to discover videos, then fetch transcripts with the YouTube Transcript Fetcher

## Workflow

1. **Configure**: Enter search terms and set result count
2. **Fetch**: Retrieve video metadata from YouTube
3. **Review**: Preview results in a table
4. **Export**: Download structured data for further analysis

## Applications

This notebook supports research that involves analyzing video media — discovering content on specific topics, mapping what videos exist in a field, tracking channel activity, identifying lecture and conference talk recordings, and building video corpora for media analysis. The structured exports provide URLs that can be used with the YouTube Transcript Fetcher to extract full transcripts.

## Methodological Positioning

This notebook represents a **computational anthropology** approach — using programmatic data retrieval to support media discovery in qualitative research. The notebook collects and structures video metadata but does not interpret it. Analytical judgment remains with the researcher.

**Important**: Respect YouTube's terms of service when using video data in research. This notebook retrieves publicly available search result metadata.

## Target Audience

Designed for anthropologists and qualitative researchers who need to discover and catalog video content — from graduate students building media corpora to research teams conducting discourse analysis across video platforms.

## Technical Approach

The notebook uses the **scrapetube** library to query YouTube search results. It supports keyword search with configurable result limits. All processing runs locally in the notebook.

## AI Anthropology Toolkit

This notebook is part of the [AI Anthropology Toolkit](https://github.com/MattArtzAnthro/AI-Anthropology-Toolkit), a collection of notebooks supporting computational and AI-assisted approaches to anthropological research.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.



## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2026. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

## Setup and Installation

Install required Python packages and import libraries for YouTube search, data processing, and interactive configuration. Run this cell first to ensure all dependencies are available.

In [ ]:
# Install required packages
!pip install scrapetube pandas openpyxl ipywidgets -q

import re
import unicodedata
import scrapetube
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


def normalize_text(text):
    """Normalize unicode characters."""
    if not isinstance(text, str):
        return text
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u2011', '-').replace('\u2013', '-').replace('\u2014', '-')
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2026', '...')
    return text


def make_slug(query):
    """Create a clean filename slug."""
    slug = re.sub(r'[^a-zA-Z0-9]+', '_', query).strip('_')[:30]
    return slug if slug else 'search'


def extract_video_data(r):
    """Extract structured data from a scrapetube result."""
    title = ''
    if isinstance(r.get('title'), dict):
        runs = r['title'].get('runs', [{}])
        title = runs[0].get('text', '') if runs else ''

    channel = ''
    if isinstance(r.get('ownerText'), dict):
        runs = r['ownerText'].get('runs', [{}])
        channel = runs[0].get('text', '') if runs else ''

    views = ''
    if isinstance(r.get('viewCountText'), dict):
        views = r['viewCountText'].get('simpleText', '')

    duration = ''
    if isinstance(r.get('lengthText'), dict):
        duration = r['lengthText'].get('simpleText', '')

    published = ''
    if isinstance(r.get('publishedTimeText'), dict):
        published = r['publishedTimeText'].get('simpleText', '')

    description = ''
    if isinstance(r.get('detailedMetadataSnippets'), list) and r['detailedMetadataSnippets']:
        snippet = r['detailedMetadataSnippets'][0]
        if isinstance(snippet.get('snippetText'), dict):
            runs = snippet['snippetText'].get('runs', [])
            description = ' '.join(run.get('text', '') for run in runs)

    vid_id = r.get('videoId', '')

    return {
        'title': normalize_text(title),
        'channel': normalize_text(channel),
        'views': views,
        'duration': duration,
        'published': published,
        'description': normalize_text(description),
        'video_id': vid_id,
        'url': f'https://youtube.com/watch?v={vid_id}' if vid_id else '',
    }


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    header_fill = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    header_font = Font(bold=True, color='FFFFFF')
    thin_border = Border(
        left=Side(style='thin', color='E7ECEF'), right=Side(style='thin', color='E7ECEF'),
        top=Side(style='thin', color='E7ECEF'), bottom=Side(style='thin', color='E7ECEF'),
    )
    for ws in wb.worksheets:
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = thin_border
        for col in ws.columns:
            max_len = max(len(str(c.value or '')) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(max_len + 2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
                cell.border = thin_border
    wb.save(filepath)


print("\u2713 All packages installed and libraries loaded successfully")
print(f"\u2713 Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print("\U0001f3ac Ready to configure your YouTube video search!")

## Search Configuration

Configure your YouTube search using the interactive controls below. Set your search terms, result count, and export format.

In [ ]:
# Interactive Search Interface

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

instructions_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
instructions_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f3ac YouTube Video Search</h2>'
instructions_html += '<p><strong>Welcome!</strong> This notebook searches YouTube and exports structured video metadata for media research.</p>'
instructions_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
instructions_html += '<ol>'
instructions_html += '<li><strong>Search:</strong> Enter keywords for your video search</li>'
instructions_html += '<li><strong>Configure:</strong> Set result count and export format</li>'
instructions_html += '<li><strong>Fetch:</strong> Click the button to retrieve video metadata</li>'
instructions_html += '<li><strong>Export:</strong> Download as CSV or Excel</li>'
instructions_html += '</ol>'
instructions_html += '<p style="margin-top: 10px; color: #8B8C89; font-size: 0.9em;">'
instructions_html += 'Use the exported video URLs with the YouTube Transcript Fetcher to get full transcripts.</p>'
instructions_html += '</div>'

instructions = widgets.HTML(value=instructions_html)

search_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f50d Search</h3>')

query_input = widgets.Text(
    value='',
    placeholder='Enter search terms (e.g., "ethnographic methods interview")',
    description='Keywords:',
    layout=layout,
    style=style
)

options_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Options</h3>')

max_results_input = widgets.IntSlider(
    value=50, min=10, max=200, step=10,
    description='Max results:',
    style=style
)

export_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f4c1 Export</h3>')

format_input = widgets.Dropdown(
    options=[('CSV (.csv)', 'csv'), ('Excel (.xlsx)', 'xlsx')],
    value='csv',
    description='Format:',
    layout=layout,
    style=style
)

fetch_btn = widgets.Button(
    description='Search YouTube',
    button_style='',
    layout=widgets.Layout(width='200px', margin='20px 0'),
    style={'button_color': '#6096BA'}
)

progress_label = widgets.HTML(value='')
out = widgets.Output()


def on_fetch_clicked(_):
    out.clear_output()
    progress_label.value = ''

    query = query_input.value.strip()
    if not query:
        with out:
            print("\u26a0\ufe0f Please enter search terms.")
        return

    max_count = max_results_input.value

    with out:
        print(f"\U0001f3ac Searching YouTube for: {query}")
        print(f"   Max results: {max_count}")
        print()

    progress_label.value = '<span style="color: #6096BA;">Fetching results from YouTube...</span>'

    try:
        videos = scrapetube.get_search(query, limit=max_count)
        all_rows = []
        for i, v in enumerate(videos):
            all_rows.append(extract_video_data(v))
            if (i + 1) % 10 == 0:
                progress_label.value = f'<span style="color: #274C77;">\u2713 {i+1} videos found...</span>'

        progress_label.value = ''

        if not all_rows:
            with out:
                print("\u26a0\ufe0f No results found. Try different search terms.")
                print("   An empty result can also mean YouTube is blocking automated requests")
                print("   from this IP address (common on Colab and other cloud IPs).")
                print("   Wait a few minutes and retry, or run the notebook locally.")
            return

        df = pd.DataFrame(all_rows)

        with out:
            print(f"\u2713 Retrieved {len(df)} videos")
            print()
            print("\U0001f4ca Preview:")
            display(df[['title', 'channel', 'views', 'duration', 'published']].head(15))

        # Export (outside widget output context)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        slug = make_slug(query)
        base = f'youtube_search_{slug}_{timestamp}'
        fmt = format_input.value

        if fmt == 'xlsx':
            filepath = f'{base}.xlsx'
            df.to_excel(filepath, sheet_name='YouTube Results', index=False, engine='openpyxl')
            style_excel(filepath)
        else:
            filepath = f'{base}.csv'
            df.to_csv(filepath, index=False)

        with out:
            print()
            print(f"\u2713 Saved: {filepath} ({len(df)} videos)")
            if IN_COLAB:
                colab_files.download(filepath)

    except Exception as e:
        progress_label.value = ''
        with out:
            print(f"\u274c Error: {type(e).__name__}: {e}")
            print("   This usually means YouTube changed its internal page structure (which breaks")
            print("   the scrapetube library) or is rate-limiting this IP address (common on Colab")
            print("   and other cloud IPs). Wait and retry, run the notebook locally, or check for")
            print("   a scrapetube update.")


fetch_btn.on_click(on_fetch_clicked)

display(widgets.VBox([
    instructions,
    search_header,
    query_input,
    options_header,
    max_results_input,
    export_header,
    format_input,
    fetch_btn,
    progress_label,
    out,
]))